### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="eryhemato_squamous_disease",
    dataset_year="1997",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5FK5P",
    download_description="""
We take the original data from UCI.

wget https://archive.ics.uci.edu/static/public/33/dermatology.zip && unzip dermatology.zip && rm dermatology.zip dermatology.names
mkdir -p local-data-warehouse/eryhemato_squamous_disease && mv dermatology.data local-data-warehouse/eryhemato_squamous_disease/
""",
    # References
    academic_reference_bibtex=r"""@article{guvenir1998learning,
  title={Learning differential diagnosis of erythemato-squamous diseases using voting feature intervals},
  author={G{\"u}venir, H Altay and Demir{\"o}z, G{\"u}l{\c{s}}en and Ilter, Nilsel},
  journal={Artificial intelligence in medicine},
  volume={13},
  number={3},
  pages={147--165},
  year={1998},
  publisher={Elsevier}
}
""",
    academic_reference_bibtex_key="guvenir1998learning",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
We start with the data from UCI.

- We encode all features but age as categorical, since they are ordinal features in nature.
- We ensure missing values in age are encoded as NaN.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="class",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss",
    stratify_on="class",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np
feature_names = ["erythema","scaling","definite borders","itching","koebner phenomenon","polygonal papules","follicular papules","oral mucosal involvement","knee and elbow involvement","scalp involvement","family history","melanin incontinence","eosinophils in the infiltrate","PNL infiltrate","fibrosis of the papillary dermis","exocytosis","acanthosis","hyperkeratosis","parakeratosis","clubbing of the rete ridges","elongation of the rete ridges","thinning of the suprapapillary epidermis","spongiform pustule","munro microabcess","focal hypergranulosis","disappearance of the granular layer","vacuolisation and damage of basal layer","spongiosis","saw-tooth appearance of retes","follicular horn plug","perifollicular parakeratosis","inflammatory monoluclear inflitrate","band-like infiltrate","age", "class"]
df = pd.read_csv(dataset_mold.path / "dermatology.data", header=None, names=feature_names)
print("Loaded data shape:", df.shape)

as_cat_col = list(df.columns)
as_cat_col.remove("age") # age is numeric, and has some missing values, so
df[as_cat_col] = df[as_cat_col].astype("category")
df["age"] = df["age"].replace("?", np.nan).astype(float)

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Loaded data shape: (366, 35)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 366
Columns: 35
Use sampling: False (sample size: 366)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['age', 'erythema', 'definite borders', 'scaling', 'itching', 'koebner phenomenon', 'oral mucosal involvement', 'knee and elbow involvement', 'polygonal papules', 'follicular papules']
Rows remaining as candidates after top-10 filter: 8 (of 366)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,erythema,scaling,definite borders,itching,koebner phenomenon,polygonal papules,follicular papules,oral mucosal involvement,knee and elbow involvement,scalp involvement,family history,melanin incontinence,eosinophils in the infiltrate,PNL infiltrate,fibrosis of the papillary dermis,exocytosis,acanthosis,hyperkeratosis,parakeratosis,clubbing of the rete ridges,elongation of the rete ridges,thinning of the suprapapillary epidermis,spongiform pustule,munro microabcess,focal hypergranulosis,disappearance of the granular layer,vacuolisation and damage of basal layer,spongiosis,saw-tooth appearance of retes,follicular horn plug,perifollicular parakeratosis,inflammatory monoluclear inflitrate,band-like infiltrate,age,class
0,2,2,2,0,0,0,0,0,2,2,1,0,0,2,0,0,2,0,3,3,2,2,2,1,0,2,0,0,0,0,0,2,0,18.0,1
1,2,2,1,0,0,0,0,0,1,0,1,0,0,2,0,0,2,1,2,2,1,2,0,1,0,0,0,0,0,0,0,0,0,NaN,1
2,1,1,0,1,3,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,0,0,0,0,0,0,2,0,0,0,2,0,40.0,4
3,3,2,2,0,0,0,0,0,0,1,1,0,0,0,0,0,3,0,2,2,3,2,0,1,0,2,0,0,0,0,0,2,0,50.0,1
4,2,1,1,3,0,3,0,1,0,0,0,1,0,0,0,2,2,0,1,0,0,0,0,0,1,0,3,0,1,0,0,2,2,29.0,3


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,erythema,category,0.0,0.00,4.0,"2, 3, 1, 0"
1,scaling,category,0.0,0.00,4.0,"2, 1, 3, 0"
2,definite borders,category,0.0,0.00,4.0,"2, 1, 0, 3"
3,itching,category,0.0,0.00,4.0,"0, 2, 3, 1"
4,koebner phenomenon,category,0.0,0.00,4.0,"0, 1, 2, 3"
5,polygonal papules,category,0.0,0.00,4.0,"0, 2, 3, 1"
6,follicular papules,category,0.0,0.00,4.0,"0, 2, 1, 3"
7,oral mucosal involvement,category,0.0,0.00,4.0,"0, 2, 3, 1"
8,knee and elbow involvement,category,0.0,0.00,4.0,"0, 2, 1, 3"
9,scalp involvement,category,0.0,0.00,4.0,"0, 2, 1, 3"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
age,358.0,36.296089,15.324557,0.0,75.0


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column                                   rank                    
PNL infiltrate                           1        0    235  64.21
                                         2        1     69  18.85
                                         3        2     55  15.03
                                         4        3      7   1.91
acanthosis                               1        2    210  57.38
                                         2        3     75  20.49
                                         3        1     71  19.40
                                         4        0     10   2.73
band-like infiltrate                     1        0    289  78.96
                                         2        3     52  14.21
                                         3        2     22   6.01
                                         4        1      3   0.82
class                                    1        1    112  30.60
                                         2        3     72  19.67
                                         3        2     61  16.67
                                         4        5     52  14.21
                                         5        4     49  13.39
clubbing of the rete ridges              1        0    252  68.85
                                         2        2     61  16.67
                                         3        3     34   9.29
                                         4        1     19   5.19
definite borders                         1        2    168  45.90
                                         2        1     93  25.41
                                         3        0     59  16.12
                                         4        3     46  12.57
disappearance of the granular layer      1        0    273  74.59
                                         2        2     49  13.39
                                         3        1     30   8.20
                                         4        3     14   3.83
elongation of the rete ridges            1        0    198  54.10
                                         2        2     95  25.96
                                         3        3     50  13.66
                                         4        1     23   6.28
eosinophils in the infiltrate            1        0    324  88.52
                                         2        1     33   9.02
                                         3        2      9   2.46
erythema                                 1        2    215  58.74
                                         2        3     90  24.59
                                         3        1     57  15.57
                                         4        0      4   1.09
exocytosis                               1        2    129  35.25
                                         2        0    118  32.24
                                         3        3     62  16.94
                                         4        1     57  15.57
family history                           1        0    320  87.43
                                         2        1     46  12.57
fibrosis of the papillary dermis         1        0    312  85.25
                                         2        2     23   6.28
                                         3        3     23   6.28
                                         4        1      8   2.19
focal hypergranulosis                    1        0    295  80.60
                                         2        2     43  11.75
                                         3        3     15   4.10
                                         4        1     13   3.55
follicular horn plug                     1        0    344  93.99
                                         2        1     10   2.73
                                         3        2      8   2.19
                                         4        3      4   1.09
follicular papules                       1        0    333  90.98
                    

In [8]:
# Target Distribution
target_df

,count,pct
class,,
1,112,30.60
3,72,19.67
2,61,16.67
5,52,14.21
4,49,13.39
6,20,5.46


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=20, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019c7246-931d-7a17-a68a-14d36719b166
2947a4bd8dd86c37af7a617e9870f5d73bef60a347551aebc751adca67b4c572
